In [1]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio

from tqdm import tqdm
from rasterio.mask import mask
from rasterio.warp import reproject
from rasterio.enums import Resampling

##########################################################################
# PATHS
##########################################################################

ROOT_DIR = Path("..")

INDICES_DIR = ROOT_DIR / "indices_aligned"

OUTPUT_DIR = ROOT_DIR / "stress_level"

OUTPUT_DIR.mkdir(exist_ok=True)

##########################################################################
# AOI
##########################################################################

crop_aoi = gpd.read_file(
    ROOT_DIR / "aoi" / "crop_boundary.gpkg"
)

##########################################################################
# YEARS
##########################################################################

YEARS = [
    2025,
    2026
]

##########################################################################
# COLORMAP
##########################################################################

STRESS_CMAP = plt.get_cmap(
    "RdYlGn_r"
).copy()

STRESS_CMAP.set_bad(
    color="lightgray"
)

In [2]:
##########################################################################
# READ REFERENCE RASTER (NDVI)
##########################################################################

def read_reference_raster(raster_path):

    with rasterio.open(raster_path) as src:

        # Reproject crop boundary if needed
        if crop_aoi.crs != src.crs:
            crop = crop_aoi.to_crs(src.crs)
        else:
            crop = crop_aoi

        # Clip
        clipped, transform = mask(
            src,
            crop.geometry,
            crop=True
        )

        data = clipped[0].astype(np.float32)

        # Handle nodata
        if src.nodata is not None:
            data[data == src.nodata] = np.nan

        data[~np.isfinite(data)] = np.nan

        profile = src.profile.copy()

        profile.update(
            height=data.shape[0],
            width=data.shape[1],
            transform=transform,
            count=1,
            dtype="float32",
            nodata=-9999.0
        )

    return data, profile


##########################################################################
# READ AND ALIGN TO REFERENCE GRID
##########################################################################

def read_aligned_raster(raster_path, reference_profile):

    with rasterio.open(raster_path) as src:

        if crop_aoi.crs != src.crs:
            crop = crop_aoi.to_crs(src.crs)
        else:
            crop = crop_aoi

        clipped, transform = mask(
            src,
            crop.geometry,
            crop=True
        )

        source = clipped[0].astype(np.float32)

        if src.nodata is not None:
            source[source == src.nodata] = np.nan

        source[~np.isfinite(source)] = np.nan

        destination = np.full(
            (
                reference_profile["height"],
                reference_profile["width"]
            ),
            np.nan,
            dtype=np.float32
        )

        reproject(
            source=source,
            destination=destination,
            src_transform=transform,
            src_crs=src.crs,
            dst_transform=reference_profile["transform"],
            dst_crs=reference_profile["crs"],
            dst_nodata=np.nan,
            resampling=Resampling.nearest
        )

        destination[~np.isfinite(destination)] = np.nan

    return destination

In [3]:
##########################################################################
# PROCESS ONE FLIGHT
##########################################################################

def process_flight(year, flight_folder):

    date = flight_folder.name.replace(
        "flight_",
        ""
    )

    output_year = OUTPUT_DIR / str(year)
    output_year.mkdir(exist_ok=True)

    output_flight = output_year / flight_folder.name
    output_flight.mkdir(exist_ok=True)

    stress_output = (
        output_flight /
        f"{date}_stress_score.tif"
    )

    preview_output = (
        output_flight /
        f"{date}_stress_preview.png"
    )

    # --------------------------------------------------
    # SKIP IF ALREADY PROCESSED
    # --------------------------------------------------

    if stress_output.exists():

        print(
            f"SKIPPING -> {flight_folder.name}"
        )

        return

    # --------------------------------------------------
    # INPUT FILES
    # --------------------------------------------------

    ndvi_path = (
        flight_folder /
        f"{date}_NDVI.tif"
    )

    gndvi_path = (
        flight_folder /
        f"{date}_GNDVI.tif"
    )

    ndre_path = (
        flight_folder /
        f"{date}_NDRE.tif"
    )

    temperature_path = (
        flight_folder /
        f"{date}_canopy_temperature.tif"
    )

    if not (
        ndvi_path.exists()
        and gndvi_path.exists()
        and ndre_path.exists()
        and temperature_path.exists()
    ):

        print(
            f"Missing input files -> {flight_folder.name}"
        )

        return

    # --------------------------------------------------
    # READ AND ALIGN
    # --------------------------------------------------

    ndvi, profile = read_reference_raster(
        ndvi_path
    )

    gndvi = read_aligned_raster(
        gndvi_path,
        profile
    )

    ndre = read_aligned_raster(
        ndre_path,
        profile
    )

    temperature = read_aligned_raster(
        temperature_path,
        profile
    )

    # --------------------------------------------------
    # CALCULATE SCORES
    # --------------------------------------------------

    ndvi_score = vegetation_score(ndvi)

    gndvi_score = vegetation_score(gndvi)

    ndre_score = vegetation_score(ndre)

    temperature_score_map = temperature_score(
        temperature
    )

    # --------------------------------------------------
    # FINAL STRESS SCORE
    # --------------------------------------------------

    stress = (
        ndvi_score +
        gndvi_score +
        ndre_score +
        temperature_score_map
    )

    # Keep NoData as NaN
    invalid_mask = (
        np.isnan(ndvi)
        | np.isnan(gndvi)
        | np.isnan(ndre)
        | np.isnan(temperature)
    )

    stress[invalid_mask] = np.nan

    # --------------------------------------------------
    # SAVE TIFF
    # --------------------------------------------------

    profile.update(
        dtype="float32",
        count=1,
        nodata=-9999.0
    )

    save_array = stress.copy()

    save_array[
        np.isnan(save_array)
    ] = -9999.0

    with rasterio.open(
        stress_output,
        "w",
        **profile
    ) as dst:

        dst.write(
            save_array.astype(np.float32),
            1
        )

    # --------------------------------------------------
    # SAVE PNG PREVIEW
    # --------------------------------------------------

    plt.figure(figsize=(7, 7))

    im = plt.imshow(
        stress,
        cmap=STRESS_CMAP,
        vmin=0,
        vmax=4
    )

    plt.colorbar(
        im,
        label="Stress Score (0-4)"
    )

    plt.title(
        f"{date} - Stress Score"
    )

    plt.axis("off")

    plt.savefig(
        preview_output,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(
        f"Processed -> {flight_folder.name}"
    )

In [4]:
##########################################################################
# EXTRACT VALID PIXELS
##########################################################################

def extract_valid_pixels(data):

    return data[np.isfinite(data)]


##########################################################################
# VEGETATION SCORE
#
# > P40           -> 0
# P20 < x <= P40   -> 0.5
# <= P20           -> 1
##########################################################################

def vegetation_score(data):

    valid = extract_valid_pixels(data)

    p20 = np.percentile(valid, 20)
    p40 = np.percentile(valid, 40)

    score = np.full(
        data.shape,
        np.nan,
        dtype=np.float32
    )

    score[data > p40] = 0.0

    score[
        (data > p20) &
        (data <= p40)
    ] = 0.5

    score[
        data <= p20
    ] = 1.0

    return score


##########################################################################
# TEMPERATURE SCORE
#
# < P70           -> 0
# P70 <= x < P85  -> 0.5
# >= 85          -> 1
##########################################################################

def temperature_score(data):

    valid = extract_valid_pixels(data)

    p70 = np.percentile(valid, 70)
    p85 = np.percentile(valid, 85)

    score = np.full(
        data.shape,
        np.nan,
        dtype=np.float32
    )

    score[
        data < p70
    ] = 0.0

    score[
        (data >= p70) &
        (data < p85)
    ] = 0.5

    score[
        data >= p85
    ] = 1.0

    return score

In [5]:
##########################################################################
# MAIN
##########################################################################

def main():

    for year in YEARS:

        print("\n" + "=" * 70)
        print(f"PROCESSING YEAR {year}")
        print("=" * 70)

        year_folder = INDICES_DIR / str(year)

        if not year_folder.exists():

            print(f"Folder not found: {year_folder}")

            continue

        flight_folders = sorted([
            f for f in year_folder.iterdir()
            if f.is_dir()
        ])

        print(
            f"Found {len(flight_folders)} flights."
        )

        for flight_folder in tqdm(flight_folders):

            try:

                process_flight(
                    year,
                    flight_folder
                )

            except Exception as e:

                print(
                    f"\nERROR processing "
                    f"{flight_folder.name}"
                )

                print(e)

    print("\n")
    print("=" * 70)
    print("All data processed")
    print("=" * 70)


##########################################################################
# RUN
##########################################################################

main()


PROCESSING YEAR 2025
Found 11 flights.


  9%|███████▌                                                                           | 1/11 [00:07<01:13,  7.37s/it]

Processed -> flight_2025_04_09


 18%|███████████████                                                                    | 2/11 [00:14<01:03,  7.02s/it]

Processed -> flight_2025_05_08


...

Processed -> flight_2026_05_28


 89%|██████████████████████████████████████████████████████████████████████████▋         | 8/9 [00:46<00:05,  5.81s/it]

Processed -> flight_2026_06_01


100%|████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:52<00:00,  5.84s/it]

Processed -> flight_2026_06_19


All data processed


In [6]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from matplotlib.backends.backend_pdf import PdfPages

##############################################################################
# PATHS
##############################################################################

STRESS_DIR = Path("..") / "stress_level"

YEARS = [2025, 2026]

##############################################################################
# CREATE REPORT
##############################################################################

def create_report(year):

    year_folder = STRESS_DIR / str(year)

    flights = sorted([
        f for f in year_folder.iterdir()
        if f.is_dir()
    ])

    output_pdf = (
        STRESS_DIR /
        f"stress_report_{year}.pdf"
    )

    with PdfPages(output_pdf) as pdf:

        images_per_page = 6

        for start in range(
            0,
            len(flights),
            images_per_page
        ):

            subset = flights[
                start:start + images_per_page
            ]

            fig, axes = plt.subplots(
                3,                      # rows
                2,                      # columns
                figsize=(8.27, 11.69)   # A4 portrait
            )

            axes = axes.flatten()

            # Hide all axes initially
            for ax in axes:
                ax.axis("off")

            # Fill available slots
            for i, flight in enumerate(subset):

                date = flight.name.replace(
                    "flight_",
                    ""
                )

                image_path = (
                    flight /
                    f"{date}_stress_preview.png"
                )

                if not image_path.exists():
                    continue

                img = mpimg.imread(
                    image_path
                )

                axes[i].imshow(img)

                axes[i].set_title(
                    date,
                    fontsize=10,
                    fontweight="bold"
                )

                axes[i].axis("off")

            plt.tight_layout()

            pdf.savefig(
                fig,
                dpi=300,
                bbox_inches="tight"
            )

            plt.close(fig)

    print(
        f"Saved: {output_pdf}"
    )

##############################################################################
# MAIN
##############################################################################

for year in YEARS:

    create_report(year)

print("\nREPORTS CREATED SUCCESSFULLY.")

Saved: C:\Users\admin\Desktop\data_try\stress_level\stress_report_2025.pdf
Saved: C:\Users\admin\Desktop\data_try\stress_level\stress_report_2026.pdf

REPORTS CREATED SUCCESSFULLY.
